# Exploratory Data Analysis — Car Damage Severity Dataset

**Source:** [kaggle.com/datasets/prajwalbhamere/car-damage-severity-dataset](https://www.kaggle.com/datasets/prajwalbhamere/car-damage-severity-dataset)

This notebook performs a full EDA pass on the Car Damage Severity dataset (image-only,
folder-organised JPEG, labels: `Minor` / `Moderate` / `Severe`), following the same EDA
structure used in the VehiDE dataset report (Milestone 2, Section 5):

1. Dataset setup & download
2. Dataset summary statistics
3. Class distribution
4. Missing value / corrupt file analysis
5. Duplicate analysis (exact + near-duplicate)
6. Feature extraction (dimensions, aspect ratio, file size, brightness, colour channels)
7. Feature distributions
8. Outlier analysis
9. Correlation analysis
10. Class-wise comparisons
11. Sample image grid
12. Summary of findings

Run this notebook top-to-bottom in Google Colab. Cells that need your Kaggle API
credentials are clearly marked.

## 0. Environment setup

In [ ]:
# Install/upgrade the packages we need for this EDA.
# imagehash -> perceptual hashing for near-duplicate detection
!pip install -q kaggle imagehash --upgrade

In [ ]:
import os
import io
import glob
import json
import hashlib
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
import imagehash

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 1. Dataset download

Upload your `kaggle.json` API token (Kaggle account → Settings → Create New API Token)
when prompted, then run the download + unzip cell.

In [ ]:
from google.colab import files

# Upload kaggle.json (skip this cell if it's already in /root/.kaggle/kaggle.json)
uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
for fname in uploaded:
    if fname == "kaggle.json":
        with open("/root/.kaggle/kaggle.json", "wb") as f:
            f.write(uploaded[fname])
os.chmod("/root/.kaggle/kaggle.json", 0o600)

TypeError: 'NoneType' object is not subscriptable

In [ ]:
DATASET_SLUG = "prajwalbhamere/car-damage-severity-dataset"
DOWNLOAD_DIR = "/content/car_damage_severity"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
!kaggle datasets download -d {DATASET_SLUG} -p {DOWNLOAD_DIR}
!unzip -q -o {DOWNLOAD_DIR}/*.zip -d {DOWNLOAD_DIR}

print("Top-level contents:")
for item in sorted(os.listdir(DOWNLOAD_DIR)):
    print(" -", item)

## 2. Locate class folders and build a file index

The dataset is folder-organised (one sub-folder per severity class, possibly nested
inside `train/` `val/` `test/` split folders). The cell below walks the directory tree,
auto-detects image files, and infers the class label from the parent folder name so this
notebook works regardless of the exact internal layout.

In [ ]:
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
KNOWN_CLASSES = {"minor", "moderate", "severe"}

records = []
for root, dirs, filenames in os.walk(DOWNLOAD_DIR):
    for fname in filenames:
        ext = os.path.splitext(fname)[1].lower()
        if ext not in IMG_EXTENSIONS:
            continue
        fpath = os.path.join(root, fname)
        # Infer class label: look for a path component matching a known class name
        parts = [p.lower() for p in Path(fpath).parts]
        label = next((p for p in parts if p in KNOWN_CLASSES), None)
        if label is None:
            # fall back to immediate parent folder name
            label = Path(fpath).parent.name.lower()
        records.append({"filepath": fpath, "filename": fname, "label": label})

df_files = pd.DataFrame(records)
print(f"Total image files discovered: {len(df_files)}")
df_files.head()

In [ ]:
print("Raw label values found (parent-folder based):")
print(df_files['label'].value_counts())

## 3. Dataset summary statistics

In [ ]:
summary = {
    "Total images": len(df_files),
    "Unique classes": df_files['label'].nunique(),
    "Classes": sorted(df_files['label'].unique().tolist()),
}
print(json.dumps(summary, indent=2))

class_counts = df_files['label'].value_counts()
summary_table = pd.DataFrame({
    "class": class_counts.index,
    "count": class_counts.values,
    "pct": (class_counts.values / len(df_files) * 100).round(2)
})
summary_table

## 4. Class distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=summary_table, x="class", y="count", ax=axes[0], palette="viridis")
axes[0].set_title("Class distribution (count)")
axes[0].set_xlabel("Severity class")
axes[0].set_ylabel("Number of images")
for i, v in enumerate(summary_table["count"]):
    axes[0].text(i, v + max(summary_table["count"]) * 0.01, str(v), ha="center")

axes[1].pie(summary_table["count"], labels=summary_table["class"], autopct="%1.1f%%",
            colors=sns.color_palette("viridis", len(summary_table)))
axes[1].set_title("Class distribution (%)")

plt.tight_layout()
plt.show()

imbalance_ratio = summary_table["count"].max() / summary_table["count"].min()
print(f"Imbalance ratio (largest class / smallest class): {imbalance_ratio:.2f} : 1")

## 5. Missing value / corrupt file analysis

For an image dataset, "missing values" translate to: unreadable/corrupt files, zero-byte
files, and any class folders that exist but contain no valid images.

In [ ]:
def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

corrupt_files = []
zero_byte_files = []

for fp in df_files["filepath"]:
    if os.path.getsize(fp) == 0:
        zero_byte_files.append(fp)
        continue
    if not is_valid_image(fp):
        corrupt_files.append(fp)

print(f"Zero-byte files found: {len(zero_byte_files)}")
print(f"Corrupt / unreadable files found: {len(corrupt_files)}")

df_files["is_corrupt"] = df_files["filepath"].isin(corrupt_files + zero_byte_files)
df_valid = df_files[~df_files["is_corrupt"]].reset_index(drop=True)
print(f"Valid images remaining for analysis: {len(df_valid)} / {len(df_files)}")

In [ ]:
missing_summary = pd.DataFrame({
    "Check": ["Zero-byte files", "Corrupt/unreadable files", "Total flagged", "Valid images"],
    "Count": [len(zero_byte_files), len(corrupt_files),
              len(zero_byte_files) + len(corrupt_files), len(df_valid)]
})
missing_summary

## 6. Duplicate analysis

Two passes: exact duplicates via MD5 hash of raw bytes, and near-duplicates via
perceptual hashing (pHash), consistent with the methodology used for VehiDE
(report Section 5.7).

In [ ]:
def md5_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df_valid["md5"] = df_valid["filepath"].apply(md5_hash)

dup_counts = df_valid["md5"].value_counts()
exact_dup_hashes = dup_counts[dup_counts > 1]
n_exact_dup_groups = len(exact_dup_hashes)
n_exact_dup_files = int(exact_dup_hashes.sum() - n_exact_dup_groups)  # extra copies only

print(f"Exact-duplicate groups (identical byte content): {n_exact_dup_groups}")
print(f"Redundant exact-duplicate files (excluding first occurrence in each group): {n_exact_dup_files}")

In [ ]:
# Perceptual hash near-duplicate detection (may take a few minutes on the full dataset)
def phash(path):
    try:
        with Image.open(path) as img:
            return imagehash.phash(img.convert("RGB"))
    except Exception:
        return None

# Use a deterministic sample cap to keep runtime reasonable on large datasets;
# set SAMPLE_N = None to hash every image.
SAMPLE_N = None
sample_df = df_valid if SAMPLE_N is None else df_valid.sample(n=min(SAMPLE_N, len(df_valid)), random_state=42)

hashes = {}
for fp in sample_df["filepath"]:
    h = phash(fp)
    if h is not None:
        hashes[fp] = h

hash_to_files = defaultdict(list)
for fp, h in hashes.items():
    hash_to_files[str(h)].append(fp)

near_dup_clusters = {h: files for h, files in hash_to_files.items() if len(files) > 1}
n_near_dup_clusters = len(near_dup_clusters)
n_near_dup_redundant = sum(len(files) - 1 for files in near_dup_clusters.values())

print(f"Near-duplicate clusters (identical pHash): {n_near_dup_clusters}")
print(f"Redundant near-duplicate files: {n_near_dup_redundant}")

In [ ]:
dup_summary = pd.DataFrame({
    "Duplicate type": ["Exact (MD5)", "Near-duplicate (pHash)"],
    "Groups/clusters found": [n_exact_dup_groups, n_near_dup_clusters],
    "Redundant files": [n_exact_dup_files, n_near_dup_redundant],
})
dup_summary

## 7. Feature extraction

Extract per-image numeric features that stand in for the "columns" of a tabular EDA:
width, height, aspect ratio, pixel area, file size, and mean brightness / per-channel
colour means.

In [ ]:
def extract_features(path):
    try:
        with Image.open(path) as img:
            img = img.convert("RGB")
            w, h = img.size
            arr = np.asarray(img.resize((128, 128)))  # downsample for fast stats
            r_mean, g_mean, b_mean = arr[:, :, 0].mean(), arr[:, :, 1].mean(), arr[:, :, 2].mean()
            brightness = arr.mean()
        return {
            "width": w,
            "height": h,
            "aspect_ratio": round(w / h, 4),
            "area_px": w * h,
            "file_size_kb": round(os.path.getsize(path) / 1024, 2),
            "brightness": round(float(brightness), 2),
            "r_mean": round(float(r_mean), 2),
            "g_mean": round(float(g_mean), 2),
            "b_mean": round(float(b_mean), 2),
        }
    except Exception:
        return None

feature_rows = []
for _, row in df_valid.iterrows():
    feats = extract_features(row["filepath"])
    if feats is not None:
        feats["filepath"] = row["filepath"]
        feats["label"] = row["label"]
        feature_rows.append(feats)

df_feat = pd.DataFrame(feature_rows)
print(f"Feature dataframe shape: {df_feat.shape}")
df_feat.head()

In [ ]:
df_feat.describe().T

## 8. Feature distributions

In [ ]:
numeric_cols = ["width", "height", "aspect_ratio", "area_px", "file_size_kb", "brightness"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for ax, col in zip(axes, numeric_cols):
    sns.histplot(df_feat[col], bins=40, kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=df_feat, x="width", y="height", hue="label", alpha=0.5, ax=ax, palette="viridis")
ax.set_title("Image resolution scatter (width vs height) by class")
plt.tight_layout()
plt.show()

print("Most common resolution:")
print(df_feat.groupby(["width", "height"]).size().sort_values(ascending=False).head(5))

## 9. Outlier analysis

Outliers are flagged using the IQR rule (1.5 x IQR beyond Q1/Q3) on the key numeric
features. This mirrors how bounding-box area outliers were treated for VehiDE.

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

outlier_summary = []
for col in numeric_cols:
    mask, lower, upper = iqr_outliers(df_feat[col])
    outlier_summary.append({
        "feature": col,
        "lower_bound": round(lower, 2),
        "upper_bound": round(upper, 2),
        "n_outliers": int(mask.sum()),
        "pct_outliers": round(mask.mean() * 100, 2),
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df_feat[col], ax=ax, color="lightcoral")
    ax.set_title(f"Boxplot: {col}")
plt.tight_layout()
plt.show()

## 10. Correlation analysis

In [ ]:
corr_cols = ["width", "height", "aspect_ratio", "area_px", "file_size_kb",
             "brightness", "r_mean", "g_mean", "b_mean"]
corr_matrix = df_feat[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix — image-level numeric features")
plt.tight_layout()
plt.show()

## 11. Class-wise comparisons

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=df_feat, x="label", y="brightness", ax=axes[0], palette="viridis")
axes[0].set_title("Brightness by severity class")

sns.boxplot(data=df_feat, x="label", y="area_px", ax=axes[1], palette="viridis")
axes[1].set_title("Image area (px) by severity class")

sns.boxplot(data=df_feat, x="label", y="file_size_kb", ax=axes[2], palette="viridis")
axes[2].set_title("File size (KB) by severity class")

plt.tight_layout()
plt.show()

In [ ]:
class_stats = df_feat.groupby("label")[numeric_cols].mean().round(2)
class_stats

## 12. Sample image grid

In [ ]:
classes = sorted(df_feat["label"].unique())
n_samples = 4

fig, axes = plt.subplots(len(classes), n_samples, figsize=(4 * n_samples, 4 * len(classes)))
if len(classes) == 1:
    axes = np.expand_dims(axes, 0)

for i, cls in enumerate(classes):
    sample_paths = df_feat[df_feat["label"] == cls]["filepath"].sample(
        n=min(n_samples, (df_feat["label"] == cls).sum()), random_state=1
    ).tolist()
    for j, fp in enumerate(sample_paths):
        img = Image.open(fp).convert("RGB")
        axes[i][j].imshow(img)
        axes[i][j].axis("off")
        if j == 0:
            axes[i][j].set_ylabel(cls, fontsize=12)
        axes[i][j].set_title(cls, fontsize=10)

plt.tight_layout()
plt.show()

## 13. Summary of findings

Run the cell below to auto-generate a short text summary from the statistics computed
above. Use this as the basis for the write-up in your dataset preparation report.

In [ ]:
print("=== EDA SUMMARY ===")
print(f"Total images discovered:      {len(df_files)}")
print(f"Corrupt/zero-byte files:      {len(corrupt_files) + len(zero_byte_files)}")
print(f"Valid images analysed:        {len(df_valid)}")
print(f"Exact-duplicate redundant files:  {n_exact_dup_files}")
print(f"Near-duplicate redundant files:   {n_near_dup_redundant}")
print(f"Classes: {dict(class_counts)}")
print(f"Class imbalance ratio (max:min): {imbalance_ratio:.2f} : 1")
print()
print("Feature ranges:")
print(df_feat[numeric_cols].agg(['min', 'max', 'mean', 'median']).round(2))
print()
print("Outlier counts per feature:")
print(outlier_df[['feature', 'n_outliers', 'pct_outliers']])

### Notes for write-up

- **Missing values**: reported as corrupt/unreadable/zero-byte file counts (Section 5) —
  there is no tabular "missing cell" concept for raw images.
- **Duplicate analysis**: exact duplicates caught by MD5 hash, near-duplicates caught by
  perceptual hash (Hamming distance 0 in this notebook's grouping — tighten/loosen by
  comparing hash Hamming distances directly if you need a softer threshold).
- **Outlier analysis**: IQR-based flagging on resolution, file size, and brightness.
  Extreme outliers are candidates for manual inspection before training.
- **Correlation analysis**: expect strong correlation between `width`/`height`/`area_px`
  (structural) and between `r_mean`/`g_mean`/`b_mean`/`brightness` (colour channels are
  not independent) — this is expected for image data and not a data-quality concern.
- **Class imbalance**: compare the ratio found here against the 6:1 Dent:Flat-Tyre ratio
  reported for VehiDE (Section 5.2) when writing the combined governance/ethics section.
